# 4) Naive Bayes Nedir?

Naive Bayes, Bayes Teoremi üzerine kurulu, olasılık tabanlı bir sınıflandırma algoritmasıdır. İsmindeki "Naive" (saf/naif) kelimesi, modelin yaptığı önemli bir varsayıma işaret eder.

### Bayes Teoremi (Hatırlatma)
- P(A|B) = P(B|A) × P(A) / P(B)

ML bağlamında:
- P(Sınıf|Özellikler) = P(Özellikler|Sınıf) × P(Sınıf) / P(Özellikler)

Yani: "Bu özelliklere sahip olduğuna göre, bu sınıfa ait olma olasılığı nedir?"

### "Naive" Varsayımı Nedir?
Model, tüm özelliklerin birbirinden bağımsız olduğunu varsayar — örneğin Titanic'te "cinsiyet" ile "yolcu sınıfı" arasında hiçbir ilişki yokmuş gibi davranır. Bu varsayım gerçekte tam doğru olmasa da (kadın olmakla 1. sınıfta olmak arasında bir ilişki olabilir), model bunu basitleştirme amacıyla göz ardı eder "naive/saf" ismi buradan gelir.

### Neden Yanlış Varsayıma Rağmen İyi Çalışıyor?
Modelin amacı olasılığı tam doğru hesaplamak değil, hangi sınıfın olasılığının diğerinden yüksek olduğunu bulmaktır. Bağımsızlık ihlali genelde her iki sınıfı da benzer şekilde etkilediği için, sınıflar arasındaki sıralama (hangisi daha olası) çoğunlukla doğru çıkar.

### Formül
- P(y|x1,x2,...,xn) ∝ P(y) × P(x1|y) × P(x2|y) × ... × P(xn|y)

- P(y) → Ön olasılık (prior): Hiçbir özelliğe bakmadan, genel olarak bir sınıfın olasılığı
- P(xi|y) → O sınıf içinde, ilgili özelliğin görülme olasılığı
- Bağımsızlık varsayımı sayesinde, her özelliğin katkısı ayrı ayrı hesaplanıp çarpılabilir — bu, hesaplamayı büyük ölçüde basitleştirir

### Somut Örnek Mantığı
Yeni bir gözlem geldiğinde, her sınıf için ayrı ayrı bir "skor" hesaplanır:
- Skor = (o sınıfın genel oranı) × (özelliğin o sınıf içindeki görülme oranı) × ...

Hangi sınıfın skoru daha yüksekse, tahmin o sınıf olur.

### Kullanılan Versiyon: GaussianNB
Sayısal/sürekli özellikler (yaş, bilet ücreti gibi) için, her özelliğin sınıf içinde normal dağılım izlediği varsayılır. Kategorik veriler için CategoricalNB veya MultinomialNB gibi farklı versiyonlar kullanılır.

In [1]:
import seaborn as sns
import pandas as pd
import numpy as np
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# --- Veri Yükleme ve Temizleme ---
titanic = sns.load_dataset('titanic')
titanic['age'] = titanic.groupby(['pclass', 'sex'])['age'].transform(lambda x: x.fillna(x.median()))
titanic = titanic.drop(columns=['deck'])
titanic = titanic.dropna(subset=['embark_town'])
titanic = titanic.drop(columns=['alive', 'embarked'])
titanic = pd.get_dummies(titanic, columns=['embark_town', 'sex'], drop_first=True)
titanic['adult_male'] = titanic['adult_male'].astype(int)
sinif_siralamasi = {'First': 1, 'Second': 2, 'Third': 3}
titanic['class'] = titanic['class'].map(sinif_siralamasi)
titanic = titanic.drop(columns=['who', 'class'])
bool_sutunlar = titanic.select_dtypes(include='bool').columns
titanic[bool_sutunlar] = titanic[bool_sutunlar].astype(int)

# --- Model Kurma ---
X = titanic.drop(columns=['survived'])
y = titanic['survived']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

nb_model = GaussianNB()
nb_model.fit(X_train, y_train)

y_pred = nb_model.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.8146

Confusion Matrix:
[[89 20]
 [13 56]]

Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.82      0.84       109
           1       0.74      0.81      0.77        69

    accuracy                           0.81       178
   macro avg       0.80      0.81      0.81       178
weighted avg       0.82      0.81      0.82       178



### Naive Bayes Sonuçları ve Karşılaştırma

Naive Bayes, %81.46 accuracy ile Logistic Regression'a (%82.58) çok yakın, KNN'in en iyi sonucuyla (K=15, %81.46) birebir aynı performansı gösterdi. Bu, "özellikler birbirinden bağımsız" varsayımının gerçekte tam doğru olmamasına rağmen (örneğin cinsiyet ve yolcu sınıfı muhtemelen ilişkili), modelin hâlâ rekabetçi sonuç verebildiğini gösteriyor — çünkü model için önemli olan olasılıkların mutlak doğruluğu değil, hangi sınıfın olasılığının diğerinden yüksek çıktığıdır.

Üç modelin performansı (Logistic: 0.8258, KNN: 0.8146, Naive Bayes: 0.8146) birbirine oldukça yakın — bu veri seti için hiçbiri diğerinden çarpıcı derecede üstün değil.